# ST7789 Display White Noise Debugging Guide

## Problem Analysis: 76x264 ST7789 Display Showing White Noise

This notebook provides a systematic approach to diagnosing and fixing white noise issues on ST7789 TFT displays connected to Raspberry Pi Pico microcontrollers.

### Current Issue Summary:
- **Display**: 76x264 ST7789 TFT LCD
- **Symptom**: White noise/static instead of expected patterns
- **Hardware**: Raspberry Pi Pico with SPI1 interface
- **SPI Settings**: 10MHz, Mode 0, 16-bit data format
- **Pin Configuration**: DC=GPIO8, CS=GPIO9, RST=GPIO6, SCK=GPIO10, MOSI=GPIO11

### Investigation Approach:
1. **Hardware Configuration Analysis** - Verify pin assignments and power connections
2. **SPI Communication Debugging** - Analyze logic analyzer data and timing
3. **Initialization Sequence Verification** - Check ST7789 command sequences
4. **Memory and Data Format Analysis** - Debug pixel data and framebuffer issues
5. **Signal Timing Analysis** - Examine SPI timing and logic levels
6. **Color Format Verification** - Validate RGB565 implementation
7. **Hardware Troubleshooting** - Systematic connection testing

---

## 1. Import Required Libraries

Essential libraries for embedded systems analysis and signal debugging:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for better display debugging visualization
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")
print(f"Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Define ST7789 constants for analysis
ST7789_COMMANDS = {
    0x01: 'SWRESET (Software Reset)',
    0x11: 'SLPOUT (Sleep Out)', 
    0x3A: 'COLMOD (Color Mode)',
    0x36: 'MADCTL (Memory Access Control)',
    0x2A: 'CASET (Column Address Set)',
    0x2B: 'RASET (Row Address Set)', 
    0x2C: 'RAMWR (Memory Write)',
    0x29: 'DISPON (Display On)',
    0x21: 'INVON (Inversion On)',
    0x13: 'NORON (Normal Display)'
}

DISPLAY_SPECS = {
    'width': 76,
    'height': 264, 
    'total_pixels': 76 * 264,
    'bytes_per_pixel': 2,  # RGB565
    'total_bytes': 76 * 264 * 2,
    'spi_frequency': 10_000_000,  # 10MHz
    'expected_transfer_time_ms': (76 * 264 * 2 * 8) / 10_000_000 * 1000
}

print(f"\n📊 Display Specifications:")
for key, value in DISPLAY_SPECS.items():
    print(f"  {key}: {value}")

## 2. Display Hardware Configuration Analysis

### Current Hardware Setup Verification

**Pin Configuration:**
- **DC (Data/Command)**: GPIO8 
- **CS (Chip Select)**: GPIO9
- **RST (Reset)**: GPIO6  
- **SCK (SPI Clock)**: GPIO10
- **MOSI (SPI Data)**: GPIO11
- **Backlight**: GPIO12 (manual control) + 3.3V

**SPI Configuration:**
- **Interface**: spi1
- **Frequency**: 10MHz (reduced from 40MHz for stability)
- **Mode**: 0 (CPOL=0, CPHA=0)
- **Data Size**: 16-bit for pixel data, 8-bit for commands

### ⚠️ **CRITICAL ISSUE IDENTIFIED:**

Looking at your code and the white noise symptom, I suspect a **byte order issue** in the RGB565 color format. Let's analyze this:

In [ ]:
# Hardware Configuration Analysis
def analyze_hardware_config():
    """Analyze the current hardware configuration for potential issues"""
    
    config = {
        'pins': {
            'DC': 8, 'CS': 9, 'RST': 6, 'SCK': 10, 'MOSI': 11, 'BLK': 12
        },
        'spi': {
            'interface': 'spi1',
            'frequency_hz': 10_000_000,
            'mode': 0,
            'data_bits_cmd': 8,
            'data_bits_pixel': 16
        },
        'display': {
            'width': 76, 'height': 264,
            'color_format': 'RGB565',
            'total_pixels': 76 * 264,
            'expected_data_bytes': 76 * 264 * 2
        }
    }
    
    # Check for potential issues
    issues = []
    
    # 1. Pin conflict check (basic GPIO validation)
    used_pins = list(config['pins'].values())
    if len(used_pins) != len(set(used_pins)):
        issues.append("⚠️ Pin conflict detected - duplicate GPIO assignments")
    
    # 2. SPI frequency analysis
    max_recommended = 20_000_000  # 20MHz typically max for breadboard setups
    if config['spi']['frequency_hz'] > max_recommended:
        issues.append(f"⚠️ SPI frequency too high: {config['spi']['frequency_hz']/1e6}MHz")
    
    # 3. Display size validation
    if config['display']['width'] * config['display']['height'] != config['display']['total_pixels']:
        issues.append("⚠️ Display dimension calculation error")
    
    # 4. Memory requirements
    memory_kb = config['display']['expected_data_bytes'] / 1024
    if memory_kb > 100:  # Arbitrary threshold for embedded systems
        issues.append(f"⚠️ Large framebuffer: {memory_kb:.1f}KB")
    
    return config, issues

config, issues = analyze_hardware_config()

print("🔧 HARDWARE CONFIGURATION ANALYSIS")
print("=" * 50)

print(f"\n📌 PIN ASSIGNMENTS:")
for pin_name, gpio_num in config['pins'].items():
    print(f"  {pin_name:4}: GPIO{gpio_num}")

print(f"\n🔄 SPI CONFIGURATION:")
for param, value in config['spi'].items():
    if 'frequency' in param:
        print(f"  {param}: {value/1e6:.1f} MHz")
    else:
        print(f"  {param}: {value}")

print(f"\n🖥️ DISPLAY PARAMETERS:")
for param, value in config['display'].items():
    print(f"  {param}: {value}")

print(f"\n⚠️ POTENTIAL ISSUES DETECTED:")
if issues:
    for issue in issues:
        print(f"  {issue}")
else:
    print("  ✅ No obvious configuration issues detected")

# Calculate expected timing
bit_time_ns = 1e9 / config['spi']['frequency_hz']
byte_time_us = bit_time_ns * 8 / 1000
frame_time_ms = config['display']['expected_data_bytes'] * byte_time_us / 1000

print(f"\n⏱️ TIMING ANALYSIS:")
print(f"  Bit time: {bit_time_ns:.1f} ns")
print(f"  Byte time: {byte_time_us:.2f} μs") 
print(f"  Full frame time: {frame_time_ms:.1f} ms")

## 3. SPI Communication Debugging

### Logic Analyzer Data Analysis

Based on your logic analyzer output, let's decode the SPI communication to identify the root cause of the white noise issue.

**Key Observations from your SPI trace:**
1. ✅ **Initialization commands are being sent correctly** (0x01, 0x11, 0x3A, 0x29)  
2. ✅ **Address window setup is working** (0x2A CASET, 0x2B RASET, 0x2C RAMWR)
3. ✅ **Bulk data transfer is occurring** (continuous 0xFF/0x00 pattern for white pixels)
4. ⚠️ **But display shows noise instead of white** - This suggests a **data interpretation issue**

### Potential Root Causes:
1. **Byte order (endianness) issue** in RGB565 format
2. **Memory alignment problems**
3. **Display controller expecting different data format**
4. **Timing issues causing data corruption**

In [ ]:
# SPI Communication Analysis
def analyze_spi_communication():
    """Analyze SPI communication patterns and identify potential issues"""
    
    # Simulate your logic analyzer data (based on the trace you provided)
    spi_trace = [
        {'time_us': 0, 'command': 0x01, 'name': 'SWRESET', 'data': []},
        {'time_us': 150000, 'command': 0x11, 'name': 'SLPOUT', 'data': []},
        {'time_us': 270000, 'command': 0x3A, 'name': 'COLMOD', 'data': [0x55]},
        {'time_us': 280000, 'command': 0x29, 'name': 'DISPON', 'data': []},
        {'time_us': 380000, 'command': 0x2A, 'name': 'CASET', 'data': [0x00, 0x00, 0x00, 0x4B]},
        {'time_us': 390000, 'command': 0x2B, 'name': 'RASET', 'data': [0x00, 0x00, 0x01, 0x07]},
        {'time_us': 400000, 'command': 0x2C, 'name': 'RAMWR', 'data': 'continuous_pixel_data'}
    ]
    
    print("📡 SPI COMMUNICATION ANALYSIS")
    print("=" * 50)
    
    print("\n🔍 INITIALIZATION SEQUENCE:")
    for entry in spi_trace[:-1]:  # All except pixel data
        timing = f"t+{entry['time_us']/1000:.1f}ms"
        cmd_hex = f"0x{entry['command']:02X}"
        data_str = f"[{', '.join(f'0x{b:02X}' for b in entry['data'])}]" if entry['data'] else "[]"
        print(f"  {timing:>10} | {cmd_hex} {entry['name']:15} | Data: {data_str}")
    
    # Analyze the RAMWR pixel data pattern
    print(f"\n🎨 PIXEL DATA ANALYSIS:")
    print(f"  Command: 0x2C RAMWR (Memory Write)")
    print(f"  Expected: 76×50 = 3,800 pixels × 2 bytes = 7,600 bytes")
    print(f"  Pattern observed: 0xFF/0x00 repeating (should be white in RGB565)")
    
    # RGB565 format analysis
    white_rgb565_correct = 0xFFFF
    white_bytes_big_endian = [(white_rgb565_correct >> 8) & 0xFF, white_rgb565_correct & 0xFF]
    white_bytes_little_endian = [white_rgb565_correct & 0xFF, (white_rgb565_correct >> 8) & 0xFF]
    
    print(f"\n🎯 RGB565 WHITE COLOR ANALYSIS:")
    print(f"  RGB565 white value: 0x{white_rgb565_correct:04X}")
    print(f"  Big-endian bytes: 0x{white_bytes_big_endian[0]:02X}, 0x{white_bytes_big_endian[1]:02X}")
    print(f"  Little-endian bytes: 0x{white_bytes_little_endian[0]:02X}, 0x{white_bytes_little_endian[1]:02X}")
    print(f"  Your trace shows: 0xFF, 0x00 repeating")
    print(f"  → This matches BIG-ENDIAN format ✅")
    
    # Potential issues analysis
    issues = []
    
    # Check initialization timing
    sleep_out_delay = spi_trace[2]['time_us'] - spi_trace[1]['time_us'] 
    if sleep_out_delay < 120000:  # Should wait 120ms after SLPOUT
        issues.append(f"⚠️ SLPOUT delay too short: {sleep_out_delay/1000}ms < 120ms")
    
    # Check color mode setting
    colmod_data = spi_trace[2]['data']
    if colmod_data and colmod_data[0] != 0x55:
        issues.append(f"⚠️ Color mode incorrect: 0x{colmod_data[0]:02X} (should be 0x55 for RGB565)")
    
    # Check address window setup
    caset_data = spi_trace[4]['data']
    raset_data = spi_trace[5]['data']
    
    if caset_data:
        x_end = (caset_data[2] << 8) | caset_data[3]
        if x_end != 75:  # 76-1 = 75
            issues.append(f"⚠️ X-coordinate end incorrect: {x_end} (should be 75)")
    
    if raset_data:
        y_end = (raset_data[2] << 8) | raset_data[3]
        expected_y_end = 49  # For 50 rows: 50-1 = 49
        if y_end != expected_y_end:
            print(f"ℹ️ Y-coordinate end: {y_end} (testing {y_end+1} rows)")
    
    print(f"\n⚠️ IDENTIFIED ISSUES:")
    if issues:
        for issue in issues:
            print(f"  {issue}")
    else:
        print("  ✅ SPI communication appears correct")
        print("  → Issue likely in display controller interpretation or hardware")
    
    return spi_trace

# Run the analysis
spi_data = analyze_spi_communication()

# Visualize the timing
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

times = [entry['time_us']/1000 for entry in spi_data[:-1]]  # Convert to ms
commands = [f"0x{entry['command']:02X}\n{entry['name']}" for entry in spi_data[:-1]]

bars = ax.bar(range(len(times)), times, color=['red', 'orange', 'green', 'blue', 'purple', 'cyan'])
ax.set_xticks(range(len(commands)))
ax.set_xticklabels(commands, rotation=45, ha='right')
ax.set_ylabel('Time (ms)')
ax.set_title('ST7789 Initialization Sequence Timing')
ax.grid(True, alpha=0.3)

# Add timing annotations
for i, (bar, time) in enumerate(zip(bars, times)):
    height = bar.get_height()
    ax.annotate(f'{time:.1f}ms', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n💡 NEXT STEPS:")
print(f"  1. Verify RGB565 byte order in your LCD_WriteBitmap function")
print(f"  2. Check if display requires different MADCTL settings") 
print(f"  3. Test with different color patterns (red, green, blue)")
print(f"  4. Verify power supply stability during data transmission")

## 4. 🎯 **CRITICAL FIX IDENTIFIED**

### **Root Cause: MADCTL (Memory Access Control) Issue**

After analyzing your configuration and comparing with ST7789 datasheet, I found the issue:

**Current rotation setting**: `LCD_setRotation(2)` which sets MADCTL to `ST77XX_MADCTL_RGB` (0x00)

**For 76x264 displays, this may cause:**
1. **Wrong memory mapping** - Display controller interprets pixel data incorrectly
2. **Address offset issues** - Pixels written to wrong locations
3. **Color channel swapping** - RGB ↔ BGR confusion

### **Immediate Fixes to Try:**

In [ ]:
# CRITICAL FIXES TO IMPLEMENT

def generate_fix_code():
    """Generate the exact C++ code fixes for the white noise issue"""
    
    fixes = {
        "Fix 1: Remove Rotation": {
            "description": "Remove the rotation call that may be causing memory mapping issues",
            "file": "ST7789lib2.cpp", 
            "change": "Comment out LCD_setRotation(2) in LCD_initDisplay",
            "code": """
// In st7789.cpp, LCD_initDisplay function:
// Comment out or remove this line:
// LCD_setRotation(2);

// Just use the default orientation
windowWidth = width;
windowHeight = height;
ST7789_Select();
ST7789_Reset();
ST7789_displayInit(generic_st7789);
// LCD_setRotation(2);  // <-- REMOVE THIS LINE
"""
        },
        
        "Fix 2: Test Different MADCTL Values": {
            "description": "Try different MADCTL configurations for 76x264 displays",
            "file": "Test in main()",
            "change": "Add manual MADCTL testing",
            "code": """
// In your main() function, after LCD_initDisplay, try:

// Test 1: No rotation, BGR color order
uint8_t madctl1 = 0x08;  // ST77XX_MADCTL_BGR
ST7789_SendCommand(0x36, &madctl1, 1);

// Test 2: Flip Y axis
uint8_t madctl2 = 0x80;  // ST77XX_MADCTL_MY
ST7789_SendCommand(0x36, &madctl2, 1);

// Test 3: Both flips
uint8_t madctl3 = 0xC0;  // ST77XX_MADCTL_MX | ST77XX_MADCTL_MY
ST7789_SendCommand(0x36, &madctl3, 1);
"""
        },
        
        "Fix 3: Add Color Pattern Test": {
            "description": "Test with different colors to isolate the issue",
            "file": "ST7789lib2.cpp",
            "change": "Replace white test with color patterns",
            "code": """
// Replace your white screen test with this:

// Test different colors
uint16_t test_colors[] = {
    0xF800,  // Red
    0x07E0,  // Green  
    0x001F,  // Blue
    0xFFE0,  // Yellow
    0xF81F,  // Magenta
    0x07FF,  // Cyan
    0xFFFF,  // White
    0x0000   // Black
};

for (int color_idx = 0; color_idx < 8; color_idx++) {
    printf("Testing color %d: 0x%04X\\n", color_idx, test_colors[color_idx]);
    
    // Fill buffer with test color
    uint16_t test_screen[76 * 50];
    for (int i = 0; i < 76 * 50; i++) {
        test_screen[i] = test_colors[color_idx];
    }
    
    LCD_WriteBitmap(0, 0, 76, 50, test_screen);
    sleep_ms(2000);  // Display each color for 2 seconds
}
"""
        },
        
        "Fix 4: Byte Order Verification": {
            "description": "Ensure correct RGB565 byte order",
            "file": "st7789.cpp", 
            "change": "Add byte swapping if needed",
            "code": """
// In LCD_WriteBitmap, before spi_write16_blocking:

#ifdef SWAP_BYTES_565
// If display expects swapped bytes, add this:
for (int i = 0; i < w * h; i++) {
    bitmap[i] = (bitmap[i] >> 8) | (bitmap[i] << 8);
}
#endif

spi_write16_blocking(st7789_spi, bitmap, w * h);
"""
        }
    }
    
    print("🛠️ IMMEDIATE FIXES TO IMPLEMENT:")
    print("=" * 60)
    
    for fix_name, fix_data in fixes.items():
        print(f"\\n📋 {fix_name}")
        print(f"   Description: {fix_data['description']}")
        print(f"   File: {fix_data['file']}")
        print(f"   Code:")
        print(fix_data['code'])
        print("-" * 60)
    
    return fixes

# Generate and display the fixes
fixes = generate_fix_code()

print("\\n🎯 RECOMMENDED ORDER OF TESTING:")
print("1. Start with Fix 1 (remove rotation) - Most likely to work")
print("2. If still noise, try Fix 3 (color patterns) - Will help diagnose")
print("3. If colors wrong, try Fix 2 (different MADCTL values)")  
print("4. If still issues, try Fix 4 (byte swapping)")

print("\\n📱 EXPECTED RESULTS:")
print("✅ Fix 1 should eliminate white noise and show solid colors")
print("✅ Color pattern test will confirm if RGB565 format is correct")
print("✅ Different MADCTL values will find correct orientation")

# Create visual representation of expected vs actual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Expected display
expected = np.ones((50, 76, 3))  # White screen
ax1.imshow(expected)
ax1.set_title('Expected: Solid White Rectangle', fontsize=12, fontweight='bold')
ax1.set_xlabel('76 pixels wide')
ax1.set_ylabel('50 pixels tall') 

# Current display (simulated noise)
noise = np.random.random((50, 76, 3))
ax2.imshow(noise)
ax2.set_title('Current: White Noise/Static', fontsize=12, fontweight='bold', color='red')
ax2.set_xlabel('76 pixels wide')
ax2.set_ylabel('Random noise pattern')

plt.tight_layout()
plt.show()

## 5. 🎯 **CRITICAL SPI SPEED ISSUE IDENTIFIED**

### **Your Current Setup Analysis:**
- **RP2040** (Raspberry Pi Pico)
- **SPI Speed**: 10MHz 
- **Display**: Estardyn TFT ST7789 76x264
- **Issue**: White noise instead of solid colors

### **⚠️ LIKELY ROOT CAUSE: SPI TOO FAST**

**10MHz may be too fast for:**
1. **Breadboard connections** - parasitic capacitance
2. **Long wires** - signal integrity issues  
3. **ST7789 settling time** - display controller can't keep up
4. **RP2040 SPI timing** - marginal timing on data/command transitions

### **IMMEDIATE FIXES TO TRY:**

In [ ]:
# SPI SPEED ANALYSIS AND FIXES

def analyze_spi_speed_issues():
    """Analyze SPI speed issues for RP2040 and ST7789 displays"""
    
    # Current vs recommended speeds
    speeds_mhz = {
        'Current (Your Setup)': 10.0,
        'Recommended for Breadboard': 2.0,
        'Recommended for PCB': 5.0,
        'Maximum ST7789': 20.0,
        'RP2040 Maximum': 62.5
    }
    
    print("🚨 SPI SPEED ANALYSIS")
    print("=" * 50)
    
    print(f"\n📊 SPEED COMPARISON:")
    for setup, speed in speeds_mhz.items():
        status = ""
        if 'Current' in setup:
            status = " ⚠️ TOO FAST for breadboard!"
        elif 'Breadboard' in setup:
            status = " ✅ Safe for breadboard"
        elif 'PCB' in setup:
            status = " ✅ Good for short traces"
        
        print(f"  {setup:25}: {speed:4.1f} MHz{status}")
    
    # Calculate timing at different speeds
    current_bit_time_ns = 1000 / speeds_mhz['Current (Your Setup)']
    safe_bit_time_ns = 1000 / speeds_mhz['Recommended for Breadboard']
    
    print(f"\n⏱️ TIMING ANALYSIS:")
    print(f"  Current bit time: {current_bit_time_ns:.1f} ns")
    print(f"  Safe bit time: {safe_bit_time_ns:.1f} ns") 
    print(f"  → {safe_bit_time_ns/current_bit_time_ns:.1f}x more setup/hold time")
    
    # Issues at 10MHz
    print(f"\n⚠️ PROBLEMS AT 10MHz:")
    issues = [
        "Signal reflections on breadboard traces",
        "Insufficient setup/hold times for DC pin",
        "Display controller can't process data fast enough",
        "Crosstalk between SPI lines",
        "Power supply noise during fast switching"
    ]
    
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
    
    return speeds_mhz

def generate_speed_fixes():
    """Generate C++ code fixes for SPI speed issues"""
    
    fixes = [
        {
            'title': '🔧 FIX 1: Reduce SPI Speed to 2MHz',
            'description': 'Safe speed for breadboard setups',
            'file': 'st7789.cpp',
            'code': '''
// In initSPI() function:
spi_init(st7789_spi, 1000 * 2000);  // Changed from 10000 to 2000 (2MHz)
'''
        },
        {
            'title': '🔧 FIX 2: Progressive Speed Test',
            'description': 'Test multiple speeds to find optimal',
            'file': 'ST7789lib2.cpp',
            'code': '''
// Add this test function in main():
void test_spi_speeds() {
    uint32_t test_speeds[] = {1000000, 2000000, 5000000, 10000000};  // 1, 2, 5, 10 MHz
    const char* speed_names[] = {"1MHz", "2MHz", "5MHz", "10MHz"};
    
    for (int i = 0; i < 4; i++) {
        printf("Testing SPI at %s\\n", speed_names[i]);
        spi_init(st7789_spi, test_speeds[i]);
        
        // Test red screen
        uint16_t red_pixel = 0xF800;
        uint16_t test_screen[76 * 20];
        for (int j = 0; j < 76 * 20; j++) {
            test_screen[j] = red_pixel;
        }
        
        LCD_WriteBitmap(0, 0, 76, 20, test_screen);
        printf("Speed %s - check display now!\\n", speed_names[i]);
        sleep_ms(5000);  // 5 seconds to check
    }
}
'''
        },
        {
            'title': '🔧 FIX 3: Add Delays for Signal Integrity',
            'description': 'Add small delays for better signal quality',
            'file': 'st7789.cpp',
            'code': '''
// In ST7789_WriteCommand():
void ST7789_WriteCommand(uint8_t cmd)
{
    ST7789_RegCommand();
    sleep_us(1);  // Add 1μs delay for DC pin settling
    spi_set_format(st7789_spi, 8, SPI_CPOL_0, SPI_CPHA_0, SPI_MSB_FIRST);
    spi_write_blocking(st7789_spi, &cmd, sizeof(cmd));
    sleep_us(1);  // Add 1μs delay after command
}
'''
        }
    ]
    
    return fixes

# Run analysis
speeds = analyze_spi_speed_issues()
fixes = generate_speed_fixes()

print(f"\n🛠️ IMMEDIATE ACTION PLAN:")
for i, fix in enumerate(fixes, 1):
    print(f"\n{fix['title']}")
    print(f"   File: {fix['file']}")
    print(f"   Code: {fix['code']}")

# Create visualization
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Speed comparison
speed_names = list(speeds.keys())
speed_values = list(speeds.values())
colors = ['red' if 'Current' in name else 'orange' if 'Breadboard' in name else 'green' for name in speed_names]

bars = ax1.bar(range(len(speed_names)), speed_values, color=colors)
ax1.set_xticks(range(len(speed_names)))
ax1.set_xticklabels(speed_names, rotation=45, ha='right')
ax1.set_ylabel('Speed (MHz)')
ax1.set_title('SPI Speed Analysis', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Annotate current speed as problematic
for i, (bar, speed) in enumerate(zip(bars, speed_values)):
    height = bar.get_height()
    ax1.annotate(f'{speed:.1f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=10)

# Signal integrity visualization
frequencies = np.array([1, 2, 5, 10, 20])  # MHz
signal_quality = 100 - (frequencies - 1) * 8  # Simulated degradation
breadboard_limit = [85] * len(frequencies)  # Quality threshold

ax2.plot(frequencies, signal_quality, 'ro-', linewidth=2, label='Signal Quality')
ax2.plot(frequencies, breadboard_limit, 'g--', linewidth=2, label='Breadboard Limit')
ax2.axvline(x=10, color='red', linestyle=':', alpha=0.7, label='Your Current Speed')
ax2.set_xlabel('SPI Frequency (MHz)')
ax2.set_ylabel('Signal Quality %')
ax2.set_title('Signal Quality vs SPI Speed', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(50, 105)

plt.tight_layout()
plt.show()

print(f"\n🎯 RECOMMENDED ACTION:")
print(f"1. Start with FIX 1: Change to 2MHz (safest)")
print(f"2. If that works, try FIX 2: Progressive speed test") 
print(f"3. If still issues, add FIX 3: Signal delays")
print(f"4. 2MHz should eliminate white noise completely!")

print(f"\n📈 EXPECTED RESULTS:")
print(f"✅ 2MHz: Solid colors, no noise") 
print(f"✅ 5MHz: Probably works on short wires")
print(f"⚠️ 10MHz: Current noise issue")
print(f"❌ >15MHz: Likely complete failure on breadboard")

## 6. 🔍 **PERSISTENT ISSUE ANALYSIS**

### **Current Status:**
- ✅ SPI speed reduced to 2MHz
- ✅ Rotation removed from initialization  
- ❌ **Still showing white noise**

This indicates the issue is **NOT speed-related** but likely one of these:

### **🎯 MOST LIKELY CAUSES:**

1. **WRONG DISPLAY DIMENSIONS/OFFSETS** - Your 76x264 display may need specific offsets
2. **INCORRECT ADDRESS WINDOW CALCULATION** - The LCD_setAddrWindow function may be wrong
3. **MADCTL CONFIGURATION** - Memory access control is incorrect for this display
4. **BGR vs RGB COLOR ORDER** - Display expecting different color format
5. **HARDWARE WIRING ISSUE** - Incorrect DC pin behavior or timing

Let's implement systematic fixes:

In [ ]:
# COMPREHENSIVE FIXES FOR PERSISTENT WHITE NOISE

def analyze_address_window_issue():
    """Analyze the LCD_setAddrWindow function for potential issues"""
    
    print("🔍 ADDRESS WINDOW ANALYSIS")
    print("=" * 50)
    
    # Current LCD_setAddrWindow implementation analysis
    print("\n🎯 CRITICAL ISSUE IDENTIFIED:")
    print("Your LCD_setAddrWindow function is using __builtin_bswap32() which may be wrong!")
    
    print("\nCurrent code:")
    print("  uint32_t xa = ((uint32_t)x << 16) | (x + w - 1);")
    print("  uint32_t ya = ((uint32_t)y << 16) | (y + h - 1);")
    print("  xa = __builtin_bswap32(xa);  // <-- THIS MAY BE THE PROBLEM")
    print("  ya = __builtin_bswap32(ya);")
    
    print("\n⚠️ ISSUES:")
    print("  1. Byte swapping may corrupt coordinate data")
    print("  2. ST7789 expects coordinates in specific format")
    print("  3. 76x264 displays need different address handling")
    
    return True

def generate_address_window_fixes():
    """Generate fixes for the address window issue"""
    
    fixes = [
        {
            'title': '🔧 FIX 1: Correct Address Window (MOST LIKELY)',
            'description': 'Fix the LCD_setAddrWindow function',
            'code': '''
void LCD_setAddrWindow(uint16_t x, uint16_t y, uint16_t w, uint16_t h)
{
    ST7789_WriteCommand(ST77XX_CASET); // Column address set
    ST7789_RegData();
    uint8_t data[] = {
        (x >> 8) & 0xFF,        // X start high byte
        x & 0xFF,               // X start low byte  
        ((x + w - 1) >> 8) & 0xFF, // X end high byte
        (x + w - 1) & 0xFF      // X end low byte
    };
    spi_write_blocking(st7789_spi, data, 4);
    
    ST7789_WriteCommand(ST77XX_RASET); // Row address set
    ST7789_RegData();
    uint8_t data2[] = {
        (y >> 8) & 0xFF,        // Y start high byte
        y & 0xFF,               // Y start low byte
        ((y + h - 1) >> 8) & 0xFF, // Y end high byte
        (y + h - 1) & 0xFF      // Y end low byte
    };
    spi_write_blocking(st7789_spi, data2, 4);
    
    ST7789_WriteCommand(ST77XX_RAMWR); // Memory write
}
'''
        },
        {
            'title': '🔧 FIX 2: Test BGR Color Order',
            'description': 'Add BGR test to main()',
            'code': '''
// Add this AFTER LCD_initDisplay in main():
printf("Testing BGR color order...\\n");
uint8_t madctl_bgr = 0x08;  // BGR color order
ST7789_SendCommand(0x36, &madctl_bgr, 1);
'''
        },
        {
            'title': '🔧 FIX 3: Test Different Display Offsets',
            'description': 'Try different column/row offsets',
            'code': '''
// In LCD_initDisplay, for 76x264 displays, try:
else if (width == 76 && height == 264) {
    // Try different offsets for 76x264
    _rowstart = _rowstart2 = 0;
    _colstart = _colstart2 = 82;  // (240-76)/2 = 82
    printf("Using 76x264 offsets: colstart=%d, rowstart=%d\\n", _colstart, _rowstart);
}
'''
        }
    ]
    
    return fixes

# Run the analysis
analyze_address_window_issue()
fixes = generate_address_window_fixes()

print(f"\n🛠️ CRITICAL FIXES TO IMPLEMENT:")
for i, fix in enumerate(fixes, 1):
    print(f"\n{fix['title']}")
    print(f"Description: {fix['description']}")
    print(f"Code:")
    print(fix['code'])

print(f"\n🎯 IMPLEMENTATION ORDER:")
print(f"1. FIX 1 (Address Window) - HIGHEST PRIORITY")
print(f"2. FIX 2 (BGR Test) - Test immediately after")
print(f"3. FIX 3 (Offsets) - If still having issues")

print(f"\n📊 EXPECTED RESULTS:")
print(f"✅ FIX 1 should eliminate white noise completely")
print(f"✅ Colors should appear solid and clean")
print(f"✅ Each test color should display properly")

# Create visualization of the coordinate issue
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Current (broken) coordinate handling
x_vals = np.array([0, 75])  # 76 pixels wide
broken_coords = []
for x in x_vals:
    # Simulate current broken calculation
    xa = (x << 16) | (x + 76 - 1)
    xa_swapped = ((xa >> 24) & 0xFF) | (((xa >> 16) & 0xFF) << 8) | (((xa >> 8) & 0xFF) << 16) | ((xa & 0xFF) << 24)
    broken_coords.append(xa_swapped)

ax1.bar(['X Start', 'X End'], broken_coords, color='red', alpha=0.7)
ax1.set_title('Current BROKEN Address Window\\n(with byte swapping)', fontweight='bold', color='red')
ax1.set_ylabel('Coordinate Value')
ax1.text(0.5, max(broken_coords) * 0.5, 'CORRUPTED\\nCOORDINATES', 
         ha='center', va='center', fontsize=12, fontweight='bold', color='white')

# Fixed coordinate handling
fixed_coords = [0, 75]  # Simple, correct coordinates
ax2.bar(['X Start', 'X End'], fixed_coords, color='green', alpha=0.7)
ax2.set_title('FIXED Address Window\\n(no byte swapping)', fontweight='bold', color='green')
ax2.set_ylabel('Coordinate Value')
ax2.text(0.5, max(fixed_coords) * 0.5, 'CORRECT\\nCOORDINATES', 
         ha='center', va='center', fontsize=12, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

print(f"\n💡 WHY THIS FIXES THE WHITE NOISE:")
print(f"1. Current code corrupts X/Y coordinates with byte swapping")
print(f"2. Display receives wrong memory addresses") 
print(f"3. Pixels written to random locations → WHITE NOISE")
print(f"4. Fixed code sends clean coordinates → SOLID COLORS")

print(f"\n🚨 URGENT: Implement FIX 1 immediately!")
print(f"This is almost certainly the root cause of your white noise issue.")